In [ ]:
import pandas as pd

df = pd.read_csv('../data/learn_dataset.csv')
print(df.head())  # Shows the first 5 rows
print(df.columns) # Shows all column names


Join with city_adm to get dep number

In [ ]:
import pandas as pd


city_adm = pd.read_csv('../data/city_adm.csv')

# Merge on 'insee_code'
merged = pd.merge(df, city_adm[['insee_code', 'dep']], on='insee_code', how='left')

# Show result
print(merged.head())

In [ ]:
print(merged['dep'].value_counts())

In [ ]:
merged['dep'].value_counts().plot(kind='bar', figsize=(12,4), title='Number of Records per Department')

In [ ]:
from scipy.stats import ks_2samp

# Ensure 'target' and 'dep' are present
departments = merged['dep'].dropna().unique()
ks_results = []

for dep in departments:
    target_in_dep = merged.loc[merged['dep'] == dep, 'target']
    target_elsewhere = merged.loc[merged['dep'] != dep, 'target']
    if len(target_in_dep) > 0 and len(target_elsewhere) > 0:
        stat, p_value = ks_2samp(target_in_dep, target_elsewhere)
        ks_results.append({'dep': dep, 'ks_stat': stat, 'p_value': p_value, 'n_in_dep': len(target_in_dep)})
        
# Convert results to DataFrame for easy viewing
ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)
print(ks_df.head(10))  # Show top 10 departments with largest KS statistic

In [ ]:
from scipy.stats import ks_2samp
import pandas as pd

# Assuming 'merged' is your DataFrame with 'dep' and 'Target' columns
departments = merged['dep'].dropna().unique()
ks_results = []

for dep in departments:
    target_in_dep = merged.loc[merged['dep'] == dep, 'target']
    target_elsewhere = merged.loc[merged['dep'] != dep, 'target']
    if len(target_in_dep) > 0 and len(target_elsewhere) > 0:
        stat, p_value = ks_2samp(target_in_dep, target_elsewhere)
        ks_results.append({'dep': dep, 'ks_stat': stat, 'p_value': p_value, 'n_in_dep': len(target_in_dep)})

# Convert results to DataFrame for easy viewing
ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)

# Show the top 10 departments with the largest KS statistic
# print(ks_df.head(10))

import plotly.express as px

# Sort by KS statistic for better visualization
ks_df_sorted = ks_df.sort_values('ks_stat', ascending=False)

fig = px.bar(
    ks_df_sorted,
    x='dep',
    y='ks_stat',
    hover_data=['p_value', 'n_in_dep'],
    labels={'dep': 'Department', 'ks_stat': 'KS Statistic'},
    title='KS Statistic by Department (Target Distribution Difference)'
)
fig.update_layout(xaxis={'categoryorder':'total descending'})
fig.show()

In [ ]:
threshold = 0.2  # Choose based on your results
group_1 = ks_df[ks_df['ks_stat'] >= threshold]['dep'].tolist()
group_2 = ks_df[ks_df['ks_stat'] < threshold]['dep'].tolist()

print("Group 1 (most different):", group_1)
print("Group 2 (more similar):", group_2)

In [ ]:
significant = ks_df[ks_df['p_value'] < 0.05]['dep'].tolist()
not_significant = ks_df[ks_df['p_value'] >= 0.05]['dep'].tolist()

print("Significantly different departments:", significant)
print("Not significantly different departments:", not_significant)

In [ ]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Option 1: Threshold-Based Grouping
threshold = 0.2  # Adjust as needed
ks_df['group_threshold'] = ks_df['ks_stat'] >= threshold

# Option 2: Significance-Based Grouping
ks_df['group_significance'] = ks_df['p_value'] < 0.05

# Prepare subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("KS Statistic Threshold", "Significance (p < 0.05)"))

# Plot for Option 1
fig.add_trace(
    go.Bar(
        x=ks_df['dep'],
        y=ks_df['ks_stat'],
        marker_color=ks_df['group_threshold'].map({True: 'crimson', False: 'royalblue'}),
        name='KS Statistic (Threshold)'
    ),
    row=1, col=1
)

# Plot for Option 2
fig.add_trace(
    go.Bar(
        x=ks_df['dep'],
        y=ks_df['ks_stat'],
        marker_color=ks_df['group_significance'].map({True: 'crimson', False: 'royalblue'}),
        name='KS Statistic (Significance)'
    ),
    row=1, col=2
)

fig.update_layout(
    title_text='Department Grouping by KS Statistic and Significance',
    xaxis_title='Department',
    yaxis_title='KS Statistic',
    showlegend=False,
    width=1100,
    height=500
)

fig.show()

On va choisir la methode 1 avec le seuil

In [ ]:
import plotly.graph_objs as go

# Option 1: Threshold-Based Grouping
threshold = 0.2  # Adjust as needed
ks_df['group'] = ks_df['ks_stat'] >= threshold

# Group 1: Individual departments above threshold
group1 = ks_df[ks_df['group']]
# Group 2: All other departments grouped together
group2 = ks_df[~ks_df['group']]

# For Group 2, aggregate KS stats (e.g., mean or sum)
group2_agg = {
    'dep': 'Group 2',
    'ks_stat': group2['ks_stat'].mean(),  # or sum, or median
    'count': len(group2)
}

# Prepare data for plotting
x = group1['dep'].tolist() + [group2_agg['dep']]
y = group1['ks_stat'].tolist() + [group2_agg['ks_stat']]

# Plot
fig = go.Figure()

# Plot Group 1 departments
fig.add_trace(go.Bar(
    x=group1['dep'],
    y=group1['ks_stat'],
    name='Group 1 (Above Threshold)',
    marker_color='crimson'
))

# Plot Group 2 as a single bar
fig.add_trace(go.Bar(
    x=[group2_agg['dep']],
    y=[group2_agg['ks_stat']],
    name=f'Group 2 (Others, n={group2_agg["count"]})',
    marker_color='royalblue'
))

fig.update_layout(
    title='KS Statistic: Group 1 Departments vs Group 2 (Others)',
    xaxis_title='Department / Group',
    yaxis_title='KS Statistic',
    width=900,
    height=500,
    showlegend=True
)
fig.show()